# D2.1 · The detection data lake — where agent telemetry lands

**Function D — The Agentic SOC → Detect — the Lake, and Rules Mapped to MITRE**

Builds on **[D1.3 · Bonus — finding the agents, and keeping what they emit](https://spbreed.github.io/cyber-commons/lessons/D1.3.html)**.

| | |
|---|---|
| Tools used | OpenSearch, OpenTelemetry |

## What this lesson is

**What it covers.** Deriving a storage tier for each telemetry source from the fastest query that reads it, and pricing that against indexing everything hot.

**Why a security engineer needs it.** The lake is designed twice — once on a whiteboard and once when the invoice arrives — and the second design is made by somebody with no information about what the SOC asks. Deriving the tier from the queries collapses that into one decision, and it is what keeps agent prompts alive: they are the biggest source, read by one query that can wait hours, and the first line cut when nobody has priced them properly.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The lake gets designed twice: once on a whiteboard where everything is indexed, and once when the invoice arrives and retention is cut across the board. The second design is the one that runs, and it is made by somebody who does not know which source a forensic replay needs.

> **At CyberTravels.** The six sources are CyberTravels' own, and the one that decides the lesson is its agent prompts: 23% of the volume, read by exactly one query, and the first thing an infrastructure review proposes deleting. Delete it and D5.1 cannot replay the refund incident at all.

## 2 · The framework

```
   the queries decide the tier. nothing else does.

   triage      "what did this agent do in the last hour"   seconds -> HOT
   scope       "everything this identity touched, 90d"     seconds -> HOT
   hunt        "unexplained tool use over a fortnight"     minutes -> WARM
   forensics   "reproduce one run, any time in a year"     hours   -> COLD
   (none)      nothing reads it                            never   -> DROP

   agent prompts: the largest source, read by ONE query, which can wait

     priced hot   ->  the line that gets cut when the bill arrives
                      and D5.1 has nothing left to replay from
     tiered cold  ->  survives at ~1% of the hot cost
```

Every rule in this chapter is written against something. This lesson is that
something, and it is designed twice if you are not careful: once on a whiteboard
where everything is indexed, and once when the bill arrives and retention is cut
across the board by whoever is holding the invoice.

The second design is the one you run, and it is made with no information about
what the SOC actually asks.

**Derive the tier from the queries.** Write down what the SOC runs and how fast
each needs an answer — triage in seconds, a hunt in minutes, a forensic replay
in hours — then tier each source by the *fastest* query that reads it. Nothing
else about the source decides it: not its volume, not how interesting it feels,
not who asked for it.

| tier | answers in | what belongs there |
|---|---|---|
| hot | seconds | anything triage or scoping reads |
| warm | minutes | anything a hunt reads |
| cold | hours | anything only forensics reads |
| drop | never | anything no query reads at all |

That last row is the one people skip. A source no query reads is not cheap
storage — it is a liability with a bill attached.

The saving is the headline. What the tiering *protects* is the point: agent
prompts are the single largest source in most estates and are read by exactly
one query, which can wait hours. Priced hot they are the line that gets cut, and
cutting them removes the only thing D5.1 can replay a run from.

> **Anchor → D1.0.** Detect is bounded below by what the lake can answer and how fast. A source tiered cold cannot serve a seconds-deep triage query however good the rule above it is — and a source deleted for cost sets that interval to infinity for everything that reads it.

## 3 · The procedure, as a skill

The skill tiers six CyberTravels sources by the fastest query that reads each, prices hot against tiered, and names the source that would have been cut.

### The skill — [`skills/detection/telemetry-tiering-cost/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/telemetry-tiering-cost/SKILL.md)

```yaml
name: telemetry-tiering-cost
description: >-
  Choose hot, warm or cold storage for each log source from the fastest SOC
  query that reads it, and price the result. Use when designing a detection
  data lake, when a SIEM bill forces a retention cut, or when agent traces are
  about to be deleted for costing too much.
allowed-tools: Read, Grep, Glob
```

# The queries decide the tier, not the retention policy

A detection lake gets designed twice. Once on a whiteboard, where everything is
indexed and searchable, and once when the bill arrives, where retention is cut
across the board by whoever is holding the invoice. The second design is the one
you run, and it is made with no information about what the SOC actually asks.

Deriving the tier from the queries makes it one decision instead of two, and it
is the difference between keeping agent traces and losing them.

## When to use this

Before building the lake, before a retention cut, and whenever a source is
proposed for deletion on cost alone.

## Procedure

**1 — Write down the queries the SOC runs, with the speed each needs.** Triage
in seconds. A hunt can take minutes. Forensic replay can take hours. If you
cannot name the query a source serves, that is the finding.

**2 — Tier each source by the fastest query that reads it.** Nothing else about
the source matters — not its volume, not how interesting it feels, not who
asked for it. Seconds means hot, minutes means warm, hours means cold, and a
source no query reads should be dropped rather than tiered.

**3 — Price both designs.** Index-everything-hot against the tiered version.
The absolute rates vary by platform; the ratios between tiers do not, and the
decision turns on the ratios.

**4 — Look at what the tiering saves, and at what it protects.** The saving is
the headline and the protection is the point: the biggest, cheapest-to-cut
source is usually agent prompts, and it is the only thing a forensic replay can
run against.

## Example

```
  index everything hot   $    18,993 / month
  tiered by query        $    13,398 / month
  difference             $     5,595 / month  (29%)
```

The run continues past this. The script is the example: `test_skills.py`
executes it on every build, so this block cannot drift from what the skill
actually prints.

## Output contract

```json
{
  "queries": [{"name": "str", "reads": ["str"], "needed_within": "seconds|minutes|hours"}],
  "sources": [{"name": "str", "gb_month": 0, "retention_days": 0,
               "tier": "hot|warm|cold|drop", "driven_by": "str"}],
  "cost": {"all_hot": 0.0, "tiered": 0.0, "saved": 0.0},
  "orphans": ["str"]
}
```

## Failure modes

- **Tiering by volume.** The biggest source is not the one that has to be fast.
- **Tiering by feeling.** "Prompts are sensitive so keep them hot" confuses
  sensitivity with latency; sensitivity is a retention and redaction decision
  (see `agent-telemetry-retention`).
- **Leaving orphans tiered.** A source no query reads is not cheap storage, it
  is a liability with a bill attached.
- **Reading the saving as the result.** The result is which sources survive the
  next cut, and why.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/telemetry-tiering-cost/scripts/telemetry_tiering_cost.py
SCRIPT = "skills/detection/telemetry-tiering-cost/scripts/telemetry_tiering_cost.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Four sources go hot because triage and scoping read them in seconds, host EDR goes warm, and agent prompts go cold — read by one query that can wait hours. Tiering costs about 29% less than indexing everything hot, and the prompts that are 23% of the volume survive at 1% of the hot price rather than being deleted.

## Your turn

List the five queries your SOC actually ran last month, then tier your sources from them. Any source that appears in no query is the finding — you are paying to store something nobody asks.

---

**Next → [D2.2 · Detections whose subject is the agent — mapped to ATT&CK and ATLAS](https://spbreed.github.io/cyber-commons/lessons/D2.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*